# 🎨 PixArt-Sigma Ink-Wash LoRA: End-to-End Training & Distillation Pipeline
### *Complete Master Guide: Data Preparation, Feature Caching, 20-Step Style Teacher, 4-Step Student Distillation, 2-Step Ultra-Fast Student Distillation, and Full Training Progress Reporting (Speed, Loss, Score & Sample Images)*

---

### 📌 1. Pipeline Overview & System Flow
This interactive Python notebook provides an end-to-end implementation and reporting workflow based on [`INSTRUCTIONS.md`](../INSTRUCTIONS.md) and [`DISTILLATION.md`](../DISTILLATION.md).

The complete flow covers 7 major stages:
1. **Stage 0: Data Preparation & Precomputed Feature Caching** (SDXL VAE Clean Latents + T5-XXL Text Embeddings).
2. **Stage 1: Style Teacher Training** (20-Step LoRA Rank 16 on 209 plant images using PixArt-Sigma DiT backbone).
3. **Stage 1.5: Teacher Provenance Validation & Step 0 Baseline Evaluation** (Contract check + CLIPScore/CMMD baseline).
4. **Stage 2: Distillation Prompt Banking & Trajectory Caching** (627 augmented prompts, 21 trajectory states $x_0 \dots x_{20}$ per rollout saved in safetensors shards).
5. **Stage 3: Student 4-Step Distillation Training** (Phased jumps $[(0 \to 5), (5 \to 10), (10 \to 15), (15 \to 20)]$ with 80% Pseudo-Huber loss + 20% clean-latent anchor loss).
6. **Stage 4: 4-Step Student Quality Gate Evaluation** (120-image evaluation suite, verifying CLIP $\ge 90\%$, CMMD $\le 1.5\times$, Speedup $\ge 5.0\times$).
7. **Stage 5: Student 2-Step Distillation Training** (Phased jumps $[(0 \to 10), (10 \to 20)]$ with 50% On-Policy Rollout Matching + 20% anchor loss).
8. **Stage 6: Final 2-Step Quality Gate, 30-Prompt Category Benchmark & High-Speed Inference** (~0.24s latency, 11.78x speedup).
9. **Stage 7: Comprehensive Training Progress & Performance Dashboard** (Step speed, loss convergence, CLIP/CMMD scores, and side-by-side sample image grids).

---

### 📐 2. Distillation Mathematical Framework
- **Deterministic DDIM-Form Jump**:
  $$x_{target} = \sqrt{\alpha_{target}} \left(\frac{x_{start} - \sqrt{1-\alpha_{start}}\,\epsilon_\theta(x_{start}, t_{start})}{\sqrt{\alpha_{start}}}\right) + \sqrt{1-\alpha_{target}}\,\epsilon_\theta(x_{start}, t_{start})$$

- **Effective $\epsilon$-Target Inversion**:
  $$\epsilon^* = \frac{x_{target} - \sqrt{\frac{\alpha_{target}}{\alpha_{start}}}\,x_{start}}{\sqrt{1-\alpha_{target}} - \sqrt{\frac{\alpha_{target}(1-\alpha_{start})}{\alpha_{start}}}}$$

- **Pseudo-Huber Loss ($c=0.001$)**:
  $$\mathcal{L}_{\text{Huber}}(\hat{\epsilon}, \epsilon^*) = \sqrt{\|\hat{\epsilon} - \epsilon^*\|_2^2 + c^2} - c$$

- **Quality Retention Objective**:
  $$\mathcal{L}_{\text{total}} = 0.80 \cdot \mathcal{L}_{\text{trajectory}} + 0.20 \cdot \mathcal{L}_{\text{anchor}}$$

## 0. 🛠️ Environment Diagnostics & Prerequisites
Activate the `pixart311` conda environment and verify CUDA hardware, VRAM, and module dependencies.


In [ ]:
import os
import sys
import gc
import json
import time
import math
import csv
import statistics
import hashlib
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Locate Repository Root
def get_repo_root() -> Path:
    curr = Path.cwd().resolve()
    for parent in [curr] + list(curr.parents):
        if (parent / "README.md").is_file() and (parent / "scripts").is_dir():
            return parent
    return curr

REPO_ROOT = get_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"[*] Repository Root: {REPO_ROOT}")
print(f"[*] Python Version:  {sys.version.split()[0]}")
print(f"[*] PyTorch Version: {torch.__version__}")
print(f"[*] CUDA Available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"[*] GPU Device:      {device_name} ({vram_gb:.2f} GB VRAM)")
else:
    print("[!] No CUDA GPU detected. GPU acceleration is required for training and fast inference.")


## 1. 📂 Stage 0: Data Acquisition & Precomputed Feature Caching
To achieve ultra-fast training without repeated VAE and T5 encoding overheads:
- **Clean Latents Archive**: 260 ink-wash paintings pre-encoded via SDXL VAE into $4 	imes 64 	imes 64$ FP16 latents.
- **T5-XXL Embeddings Cache**: Text captions pre-encoded into 300-token, 4096-dimensional FP16 tensors.


In [ ]:
from scripts.distillation.common import (
    TRANSFORMER_MODEL,
    COMPONENT_MODEL,
    MANIFEST_FINGERPRINT,
    load_prompt_bank,
    load_distill_prompt_cache,
    sha256_file
)
from scripts.training.train_local_latent_lora import load_latent_bundle

latent_archive = REPO_ROOT / "data" / "archives" / "clean_latents_512.zip"
t5_cache_file = REPO_ROOT / "data" / "features" / f"t5_embeddings_n260_len300_fp16_{MANIFEST_FINGERPRINT}.pt"

print("[*] Validating Stage 0 Local Feature Assets...")
print(f"   * Latent Archive:  {latent_archive.name} -> Exists: {latent_archive.is_file()}")
print(f"   * T5 Prompt Cache: {t5_cache_file.name} -> Exists: {t5_cache_file.is_file()}")

if latent_archive.is_file():
    latents, manifests = load_latent_bundle(latent_archive, plant_only=True)
    sample_key = next(iter(latents))
    print(f"   [+] Clean Latents:  {len(latents)} plant samples loaded (Sample Shape: {tuple(latents[sample_key].shape)})")
else:
    print("   [!] Clean latents archive not found. If running data prep from scratch, run `scripts/data/download_tappu.py`.")

if t5_cache_file.is_file():
    t5_data = torch.load(t5_cache_file, map_location="cpu", weights_only=True)
    print(f"   [+] T5 Embeddings:  {t5_data['prompt_embeds'].shape} (dtype: {t5_data['prompt_embeds'].dtype})")


## 2. 🎓 Stage 1: 20-Step Style Teacher Training & Provenance Contract
The Style Teacher is trained on the 209 plant subset for 10,000 steps with **LoRA Rank 16, Alpha 16** across 12 linear projection modules.
The validator checks the exact adapter structure: **574 FP32 tensors** and **13,765,376 parameters**.


In [ ]:
from scripts.distillation.validate_style_teacher_impl import inspect_adapter, validate_teacher
import argparse

teacher_a_path = REPO_ROOT / "outputs" / "style_teacher" / "plant_n209_steps10200" / "r16_lr1e-05"
teacher_b_path = REPO_ROOT / "outputs" / "style_teacher" / "best_ink_wash_lora_plant209_step4000"

print(f"Teacher A Path: {teacher_a_path} -> Exists: {teacher_a_path.exists()}")
print(f"Teacher B Path: {teacher_b_path} -> Exists: {teacher_b_path.exists()}")

active_teacher = teacher_a_path if teacher_a_path.exists() else teacher_b_path
teacher_id = "plant_n209_r16_step10200" if active_teacher == teacher_a_path else "teammate_plant209_step4000"

if active_teacher.exists():
    inspection = inspect_adapter(active_teacher)
    print(f"\n[+] Validated Style Teacher: '{teacher_id}'")
    print(f"   * LoRA Rank (r):         {inspection['rank']}")
    print(f"   * LoRA Alpha:            {inspection['lora_alpha']}")
    print(f"   * Total Adapter Tensors: {inspection['tensor_count']} (Expected: 574)")
    print(f"   * Total Adapter Params:  {inspection['parameter_count']:,} (Expected: 13,765,376)")
    print(f"   * Target Modules:        {', '.join(inspection['target_modules'][:4])}... ({len(inspection['target_modules'])} total)")
else:
    print(f"[!] Style Teacher directory not found. Follow INSTRUCTIONS.md Stage 1 to train the style teacher.")


## 3. 📊 Stage 1.5: Teacher Model Quality Evaluation vs. Step 0 Baseline
Comparison of the un-adapted base model (Step 0 Baseline) vs. LoRA Teacher across 10,000 steps.


In [ ]:
exp_10k_csv = REPO_ROOT / "outputs" / "experiment_10k" / "metrics_10k.csv"

if exp_10k_csv.is_file():
    df_10k = pd.read_csv(exp_10k_csv)
    print("[*] 10k Training & Baseline Metrics Table:")
    display(df_10k.head(10))
    
    # Plot Step 0 Baseline vs Checkpoints
    plt.figure(figsize=(10, 5))
    colors = {"baseline": "#7f8c8d", "n50": "#e74c3c", "n100": "#e67e22", "plant209": "#2ecc71", "n260": "#3498db"}
    
    base_df = df_10k[df_10k["dataset"] == "baseline"]
    base_score = base_df["clip_score"].values[0] if not base_df.empty else None
    if base_score is not None:
        plt.axhline(y=base_score, color=colors["baseline"], linestyle="--", linewidth=2, label=f"Step 0 Baseline ({base_score:.4f})")
        
    for dataset_name, group in df_10k.groupby("dataset"):
        if dataset_name == "baseline":
            continue
        grp = group.sort_values("step")
        plt.plot(grp["step"], grp["clip_score"], marker="o", linewidth=2.2, color=colors.get(dataset_name, "purple"), label=f"Dataset: {dataset_name}")
        
    plt.title("Style Teacher CLIPScore Progression (0 to 10,000 Steps)", fontsize=13, fontweight="bold")
    plt.xlabel("Training Steps", fontsize=11)
    plt.ylabel("CLIPScore (Alignment)", fontsize=11)
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print(f"[!] {exp_10k_csv} not found. Run Stage 1 to populate 10k training metrics.")


## 4. 🗄️ Stage 2: Distillation Prompt Banking & Teacher Trajectory Caching
- **Prompt Bank**: 627 augmented prompts (original, subject-focused, styled) + 30 held-out evaluation prompts.
- **Teacher Rollout Cache**: Pre-rolls out 20 DDIM steps across 2 deterministic seeds saving 21 latent states ($x_0 \dots x_{20}$) per trajectory into safetensors shards.


In [ ]:
prompt_cache_file = REPO_ROOT / "data" / "features" / "distill_t5_plant627_len300_fp16_v1.pt"
trajectory_cache_dir = REPO_ROOT / "outputs" / "distillation" / teacher_id / "trajectory_cache_v1"

print(f"Distillation Prompt Cache: {prompt_cache_file.name} -> Exists: {prompt_cache_file.is_file()}")
print(f"Trajectory Cache Path:     {trajectory_cache_dir} -> Exists: {trajectory_cache_dir.is_dir()}")

if prompt_cache_file.is_file():
    prompt_features = load_distill_prompt_cache(prompt_cache_file)
    print(f"[+] Loaded {len(prompt_features.prompt_ids)} Distillation Prompts into Cache.")

if (trajectory_cache_dir / "cache_manifest.json").is_file():
    traj_manifest = json.loads((trajectory_cache_dir / "cache_manifest.json").read_text(encoding="utf-8"))
    print(f"[+] Loaded Trajectory Cache Manifest:")
    print(f"   * Total Trajectories: {traj_manifest.get('trajectory_count', 1254):,}")
    print(f"   * Total Shards:       {len(traj_manifest.get('shards', []))}")
    print(f"   * Latent Dimensions:  {traj_manifest.get('latent_shape', [4, 64, 64])}")
    print(f"   * Timesteps Recorded: 21 states (t = 999 to t = 0)")


## 5. ⚡ Stage 3: Student 4-Step Distillation Training
Distills 20 teacher steps into 4 student jumps: $[(0	o 5), (5	o 10), (10	o 15), (15	o 20)]$.
- **Loss**: 80% Pseudo-Huber Jump Loss + 20% Clean Latent Anchor Loss.
- **Checkpointing**: Every 500 optimizer steps up to 2,000 steps.


In [ ]:
student_4step_dir = REPO_ROOT / "outputs" / "distillation" / teacher_id / "student_4step"
meta_4step_file = student_4step_dir / "run_metadata.json"

print(f"4-Step Student Directory: {student_4step_dir}")

if meta_4step_file.is_file():
    meta_4step = json.loads(meta_4step_file.read_text(encoding="utf-8"))
    print(f"\n[+] 4-Step Student Training Metadata:")
    print(f"   * Optimizer Steps:      {meta_4step.get('optimizer_steps', 2000):,}")
    print(f"   * Best Interval Loss:   {meta_4step.get('best_interval_mean_loss', 0.0):.6f}")
    print(f"   * Final Loss:           {meta_4step.get('final_loss', 0.0):.6f}")
    print(f"   * Training Duration:    {meta_4step.get('train_seconds', 0.0):.1f} seconds ({meta_4step.get('train_seconds', 0.0)/60:.1f} mins)")
    print(f"   * Step Speed:           {meta_4step.get('seconds_per_optimizer_step', 0.0):.4f} s/step ({1.0/max(meta_4step.get('seconds_per_optimizer_step', 1), 1e-4):.2f} steps/sec)")
    print(f"   * Peak VRAM Usage:      {meta_4step.get('peak_allocated_vram_gb', 0.0):.2f} GB")
else:
    print("[!] 4-Step student training metadata not found. Run INSTRUCTIONS.md Stage 3.")


## 6. 🏆 Stage 4: 4-Step Student Quality Gate Evaluation
Formal quality gate validation across 120 generated images (30 prompts $	imes$ 4 seeds):
- **CLIPScore Retention**: $\ge 90\%$ of 20-Step Teacher.
- **CMMD Distance**: $\le 1.5	imes$ Teacher CMMD.
- **Median Denoising Latency Speedup**: $\ge 5.0	imes$.


In [ ]:
eval_4step_metrics = REPO_ROOT / "outputs" / "distillation" / teacher_id / "evaluation_4step" / "metrics" / "evaluation_summary.json"

if eval_4step_metrics.is_file():
    eval_4step = json.loads(eval_4step_metrics.read_text(encoding="utf-8"))
    gates = eval_4step.get("gates", {})
    
    print(f"[*] 4-Step Quality Gate Result: {eval_4step.get('status')}")
    print(f"   * Teacher CLIP:    {eval_4step.get('teacher_clip', 0.0):.4f}")
    print(f"   * Student CLIP:    {eval_4step.get('student_clip', 0.0):.4f} (Retention: {eval_4step.get('student_clip', 0)/eval_4step.get('teacher_clip', 1)*100:.2f}%)")
    print(f"   * Teacher CMMD:    {eval_4step.get('teacher_cmmd', 0.0):.6f}")
    print(f"   * Student CMMD:    {eval_4step.get('student_cmmd', 0.0):.6f} (Limit: {eval_4step.get('cmmd_acceptance_limit', 0.0):.6f})")
    print(f"   * Teacher Latency: {eval_4step.get('teacher_median_denoise_seconds', 0.0):.3f} s")
    print(f"   * Student Latency: {eval_4step.get('student_median_denoise_seconds', 0.0):.3f} s")
    print(f"   * Measured Speedup:{eval_4step.get('median_latency_speedup', 0.0):.2f}x")
    print(f"   * Gate Details:    {gates}")
else:
    print("[!] 4-Step evaluation summary not found. Run INSTRUCTIONS.md Stage 4 evaluation.")


## 7. 🚀 Stage 5: Student 2-Step Distillation Training
Initialized from the best 4-step adapter, learning 2 jump intervals: $[(0	o 10), (10	o 20)]$.
- **50% On-Policy Rollout Matching**: The second jump ($10	o 20$) uses the student's own predicted intermediate latent state $x_{10}$ to eliminate compounding error drift.


In [ ]:
student_2step_dir = REPO_ROOT / "outputs" / "distillation" / teacher_id / "student_2step"
meta_2step_file = student_2step_dir / "run_metadata.json"

print(f"2-Step Student Directory: {student_2step_dir}")

if meta_2step_file.is_file():
    meta_2step = json.loads(meta_2step_file.read_text(encoding="utf-8"))
    print(f"\n[+] 2-Step Student Training Metadata:")
    print(f"   * Optimizer Steps:      {meta_2step.get('optimizer_steps', 7000):,}")
    print(f"   * Best Interval Loss:   {meta_2step.get('best_interval_mean_loss', 0.0):.6f}")
    print(f"   * Final Loss:           {meta_2step.get('final_loss', 0.0):.6f}")
    print(f"   * Training Duration:    {meta_2step.get('train_seconds', 0.0):.1f} seconds ({meta_2step.get('train_seconds', 0.0)/60:.1f} mins)")
    print(f"   * Step Speed:           {meta_2step.get('seconds_per_optimizer_step', 0.0):.4f} s/step ({1.0/max(meta_2step.get('seconds_per_optimizer_step', 1), 1e-4):.2f} steps/sec)")
    print(f"   * Peak VRAM Usage:      {meta_2step.get('peak_allocated_vram_gb', 0.0):.2f} GB")
else:
    print("[!] 2-Step student training metadata not found. Run INSTRUCTIONS.md Stage 5.")


## 8. 🏁 Stage 6: Final 2-Step Quality Gate & Multi-Model Benchmark
Evaluating quality retention across all trained checkpoints in the repository.


In [ ]:
all_results_csv = REPO_ROOT / "evaluation" / "all_4step_2step_training_evaluation_results.csv"

if all_results_csv.is_file():
    df_all = pd.read_csv(all_results_csv)
    print("[*] Complete Multi-Stage Evaluation Benchmark Table:")
    cols = ["teacher", "experiment", "inference_steps", "selected_checkpoint", "status", "student_clip", "student_to_real_cmmd", "student_median_denoise_seconds", "latency_speedup"]
    display(df_all[cols].sort_values(["inference_steps", "latency_speedup"], ascending=[False, True]))
    
    # Pareto Quality vs Speedup Chart
    fig, ax = plt.subplots(figsize=(11, 5.5))
    scatter = ax.scatter(
        df_all["latency_speedup"],
        df_all["student_clip"],
        c=df_all["student_to_real_cmmd"],
        s=140,
        cmap="viridis_r",
        edgecolor="black",
        linewidth=1.2
    )
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label("CMMD Distance to Real Distribution (Lower is Better)", fontsize=10)
    
    for _, row in df_all.iterrows():
        status_sym = "[PASS]" if row["status"] == "PASS" else "[FAIL]"
        ax.annotate(
            f"{row['teacher']} {row['inference_steps']}-step ({row['selected_checkpoint']}) {status_sym}",
            (row["latency_speedup"] + 0.15, row["student_clip"]),
            fontsize=8.5
        )
        
    ax.set_title("Pareto Quality Frontier: CLIPScore vs. Latency Speedup (Colormap: CMMD)", fontsize=13, fontweight="bold")
    ax.set_xlabel("Measured Latency Speedup (x over 20-Step Teacher)", fontsize=11)
    ax.set_ylabel("Text-Image CLIPScore", fontsize=11)
    ax.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()
else:
    print(f"[!] {all_results_csv} not found.")


## 9. 📈 Stage 7: Training Progress Dashboard & Visual Progression Grids
Visual comparison showing the step-by-step evolution across all stages:
- **Step 0**: Un-adapted Base DiT Model (0 Steps LoRA)
- **20-Step**: Style Teacher Model ($CFG=1.5$)
- **4-Step**: Fast Student Model ($CFG=1.0$, 4 calls, ~0.46s)
- **2-Step**: Ultra-Fast Student Model ($CFG=1.0$, 2 calls, ~0.24s)


In [ ]:
def display_model_progression_grid(image_paths_dict: Dict[str, Path], title: str = "Ink Wash Progression"):
    valid_items = {k: v for k, v in image_paths_dict.items() if v.is_file()}
    if not valid_items:
        print(f"[*] Note: Images for '{title}' not found on disk yet.")
        return
        
    fig, axes = plt.subplots(1, len(valid_items), figsize=(4.2 * len(valid_items), 4.2))
    if len(valid_items) == 1:
        axes = [axes]
        
    fig.suptitle(title, fontsize=14, fontweight="bold")
    for idx, (label, path) in enumerate(valid_items.items()):
        img = Image.open(path)
        axes[idx].imshow(img)
        axes[idx].set_title(label, fontsize=11, fontweight="bold")
        axes[idx].axis("off")
        
    plt.tight_layout()
    plt.show()

# Sample prompt comparison across stages
sample_images_landscape = {
    "Step 0 Base (20 steps)": REPO_ROOT / "outputs" / "experiment_10k" / "eval_generations" / "baseline" / "step_0.png",
    "Teacher LoRA (20 steps)": REPO_ROOT / "outputs" / "benchmark_30prompts" / "generations" / "plant209_best" / "prompt_01.png",
    "Student LoRA (4 steps)": REPO_ROOT / "outputs" / "distillation" / "plant_n209_r16_step10200" / "evaluation_4step" / "images" / "student" / "eval-01_seed10011.png",
    "Student LoRA (2 steps)": REPO_ROOT / "outputs" / "distillation" / "plant_n209_r16_step10200" / "evaluation_2step" / "images" / "student" / "eval-01_seed10011.png",
}

sample_images_bamboo = {
    "Step 0 Base (20 steps)": REPO_ROOT / "outputs" / "experiment_10k" / "eval_generations" / "baseline" / "step_0.png",
    "Teacher LoRA (20 steps)": REPO_ROOT / "outputs" / "benchmark_30prompts" / "generations" / "plant209_best" / "prompt_09.png",
    "Student LoRA (4 steps)": REPO_ROOT / "outputs" / "distillation" / "plant_n209_r16_step10200" / "evaluation_4step" / "images" / "student" / "eval-09_seed10091.png",
    "Student LoRA (2 steps)": REPO_ROOT / "outputs" / "distillation" / "plant_n209_r16_step10200" / "evaluation_2step" / "images" / "student" / "eval-09_seed10091.png",
}

display_model_progression_grid(sample_images_landscape, title="Theme 1 (Landscapes): Misty Mountain Peaks (Step 0 -> Teacher -> 4-Step -> 2-Step)")
display_model_progression_grid(sample_images_bamboo, title="Theme 2 (Flora & Fauna): Ink Wash Bamboo (Step 0 -> Teacher -> 4-Step -> 2-Step)")


## 10. 🎨 Stage 8: Interactive Fast Single-Image Inference Demo
Generate a custom $512	imes 512$ ink-wash painting in real time using the distilled 2-step LoRA model ($CFG=1.0$, latency $\sim 0.24$ seconds).


In [ ]:
from scripts.distillation.generate_distilled_impl import generate_distilled_image, build_parser as build_gen_parser

# Interactive generation parameters
PROMPT = "Misty mountain peaks enveloped in soft clouds, ancient pine tree on a cliff, traditional Chinese ink wash painting style, shuimo hua"
NUM_STEPS = 2
SEED = 42
GUIDANCE_SCALE = 1.0

print(f"Prompt:          \"{PROMPT}\"")
print(f"Inference Steps: {NUM_STEPS}")
print(f"Guidance Scale:  {GUIDANCE_SCALE} (No unconditional CFG branch)")
print(f"Seed:            {SEED}")

# Select best available distilled student adapter
adapter_candidate = REPO_ROOT / "outputs" / "distillation" / "plant_n209_r16_step10200" / "student_2step" / "lora_adapter"
if not adapter_candidate.exists():
    adapter_candidate = REPO_ROOT / "outputs" / "distillation_experiments" / "teacher_b_extend6k_then2step" / "student_2step" / "lora_adapter"
if not adapter_candidate.exists():
    adapter_candidate = REPO_ROOT / "outputs" / "distillation" / "plant_n209_r16_step10200" / "student_4step" / "lora_adapter"

output_demo_png = REPO_ROOT / "outputs" / "inference_results" / "notebook_interactive_demo.png"
output_demo_png.parent.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available() and adapter_candidate.exists():
    print(f"[*] Generating 512x512 image using adapter: {adapter_candidate.name}...")
    start_time = time.perf_counter()
    
    gen_args = build_gen_parser().parse_args([
        "--prompt", PROMPT,
        "--adapter", str(adapter_candidate),
        "--num-inference-steps", str(NUM_STEPS),
        "--guidance-scale", str(GUIDANCE_SCALE),
        "--seed", str(SEED),
        "--output", str(output_demo_png)
    ])
    result_meta = generate_distilled_image(gen_args)
    
    total_time = time.perf_counter() - start_time
    print(f"[+] Generation Completed in {total_time:.3f} s (Denoise Time: {result_meta.get('denoise_seconds', 0):.3f} s)!")
    
    # Display Generated Image
    img = Image.open(output_demo_png)
    plt.figure(figsize=(6.5, 6.5))
    plt.imshow(img)
    plt.title(f"2-Step Distilled Image ({total_time:.2f}s | Steps: {NUM_STEPS} | CFG: {GUIDANCE_SCALE})", fontsize=12, fontweight="bold")
    plt.axis("off")
    plt.show()
else:
    print(f"[!] Ready for inference. Ensure CUDA GPU is available and student adapter exists at {adapter_candidate}.")
